# 08 — Chạy `dev` và đóng gói bài nộp

### Quy tắc của notebook này

Tập `dev` chạy **đúng một lần**, ở đây, ở cuối. Mọi lựa chọn đã chốt xong trước
khi mở notebook này ra.

Lý do không phải là hình thức. `dev` là ước lượng không thiên lệch cho điểm
thật chỉ khi chưa có lựa chọn nào được đưa ra dựa trên nó. Chạy `dev`, thấy
điểm thấp, chỉnh tham số rồi chạy lại — sau vài vòng như vậy `dev` trở thành
một tập `train` thứ hai, và con số nó cho không còn nói được gì về tập test ẩn.

Notebook `03b` đã mô phỏng mức độ nghiêm trọng: chọn cái tốt nhất trong 20
phương án ngang tài làm điểm của người thắng phồng lên **+0,0280**. Đó là lý do
dự án chia `train` thành `fit` và `check`, và để dành `dev` cho đúng lần này.

### Việc của notebook

| Mục | Việc |
| --- | --- |
| 1 | Xác nhận lựa chọn đã chốt trên `train` |
| 2 | Chạy hệ thống đó trên `dev`, một lần |
| 3 | Đủ bộ độ đo trên `dev` |
| 4 | Xuất file TREC và kiểm định dạng |
| 5 | Danh sách mô hình phải công khai trong bài báo hệ thống |
| 6 | Bảng tổng của cả dự án |

### Luật cuộc thi cần tuân thủ

| Luật | Cách dự án làm |
| --- | --- |
| Chỉ truy hồi trong kho đã cho | Không dùng nguồn ngoài, mục 5 liệt kê đủ |
| Công khai mọi mô hình và API đã dùng | Mục 5 sinh ra danh sách đó |
| Định dạng nộp đúng chuẩn TREC | Mục 4 chạy `format_checker.py` của ban tổ chức |

In [ ]:
import sys
import json
import time
import subprocess
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path("..") / "src"))
import reteco as R
import pipeline as P

DATA = Path(r"D:\RETECO-project\reteco_data\track1_tempo")
CHECKER = Path(r"D:\RETECO-project\RETECO\starter_kit\format_checker.py")
SYSTEMS = Path("..") / "systems"
SUBMISSIONS = Path("..") / "submissions"
CACHE = Path("results")
SUBMISSIONS.mkdir(exist_ok=True)

P.configure(DATA, CACHE)
domains = P.domains()

print(f"{len(domains)} domains")
print(f"format checker present: {CHECKER.exists()}")
dev_queries = sum(1 for d in domains
                  for _ in R.load_queries(DATA / d / "examples_dev.jsonl"))
print(f"dev queries: {dev_queries:,}")

In [ ]:
# --- Shared chart style -------------------------------------------------
# Same palette and helpers as every other notebook in the project, so a bar
# here means what a bar there means.
BLUE, ORANGE, TEAL, AMBER, PINK, VIOLET = (
    "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#4a3aa7")
SURFACE, INK, INK_SOFT, INK_MUTED = "#fcfcfb", "#0b0b0b", "#52514e", "#898781"
GRID, AXIS = "#e1e0d9", "#c3c2b7"

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
    "savefig.facecolor": SURFACE, "axes.edgecolor": AXIS,
    "axes.labelcolor": INK_SOFT, "axes.labelsize": 9.5,
    "text.color": INK, "xtick.color": INK_MUTED, "ytick.color": INK_MUTED,
    "xtick.labelsize": 9.5, "ytick.labelsize": 9.5, "font.size": 10,
    "figure.dpi": 120, "axes.linewidth": 0.9,
})


def finish(ax, title, subtitle=None, xlabel=None, ylabel=None,
           grid_axis="y", note=None):
    if subtitle:
        ax.set_title(subtitle, loc="left", pad=8, fontsize=9.5, color=INK_MUTED)
        ax.annotate(title, xy=(0, 1), xycoords="axes fraction",
                    xytext=(0, 24), textcoords="offset points",
                    ha="left", va="bottom", fontsize=12.5,
                    fontweight="bold", color=INK, annotation_clip=False)
    else:
        ax.set_title(title, loc="left", pad=12, fontsize=12.5,
                     fontweight="bold", color=INK)
    ax.set_xlabel(xlabel or "")
    ax.set_ylabel(ylabel or "")
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.spines["left" if grid_axis == "x" else "bottom"].set_color(AXIS)
    if grid_axis == "x":
        ax.spines["bottom"].set_visible(False)
    elif grid_axis:
        ax.spines["left"].set_visible(False)
    if grid_axis:
        ax.grid(axis=grid_axis, color=GRID, linewidth=0.8)
        ax.set_axisbelow(True)
    ax.tick_params(length=0)
    if note:
        ax.text(0, -0.34, note, transform=ax.transAxes, ha="left", va="top",
                fontsize=8.5, color=INK_MUTED)
    return ax


def label_bars(ax, bars, values, fmt="{:,.0f}", horizontal=True, pad=0.015):
    span = max(values) if len(values) else 1
    for bar, value in zip(bars, values):
        if horizontal:
            ax.text(bar.get_width() + span * pad,
                    bar.get_y() + bar.get_height() / 2, fmt.format(value),
                    va="center", ha="left", fontsize=8.5, color=INK_SOFT)
        else:
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + span * pad, fmt.format(value),
                    ha="center", va="bottom", fontsize=8.5, color=INK_SOFT)


def legend_below(ax, ncol=2, y=-0.30):
    ax.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.5, y),
              ncol=ncol, fontsize=9.5, handlelength=1.1, handleheight=1.1,
              columnspacing=1.8)

---

## 1. Lựa chọn đã chốt

Đọc lại quyết định của `07d`. Nếu chưa chạy `07d` thì lấy hệ thống cao điểm
nhất trên `fit` trong số những hệ thống trả lời đủ câu.

In [ ]:
# --- What train chose ----------------------------------------------------
hybrid_file = CACHE / "hybrid_summary.json"
if hybrid_file.exists():
    hybrid = json.loads(hybrid_file.read_text(encoding="utf-8"))
    CHOSEN_NAME = hybrid["chosen"]["name"]
    print(f"07d chose: {CHOSEN_NAME}")
else:
    records = [r for r in P.table() if r["n_topics"] == r["n_gold_queries"]]
    CHOSEN_NAME = max(records, key=lambda r: r["fit"]["ndcg"])["name"]
    print(f"\n07d has not been run. Falling back to the highest fit score "
          f"among complete systems: {CHOSEN_NAME}")

# Find the file that defines it.
candidates = sorted(SYSTEMS.glob("*.json"))
chosen_path = None
for path in candidates:
    if P.load_system(path)["name"] == CHOSEN_NAME:
        chosen_path = path
        break
assert chosen_path, (
    f"No file in {SYSTEMS} defines a system named {CHOSEN_NAME!r}. "
    f"Found: {[p.name for p in candidates]}")

CHOSEN = P.load_system(chosen_path)
print(f"\nfile: {chosen_path}")
print(json.dumps(CHOSEN, indent=1))

In [ ]:
# --- Its record on train, for reference ----------------------------------
train_digest = P.run(CHOSEN, split="train")
train_record = P.score(train_digest)

print(f"{'metric':<16}{'train':>10}")
print("-" * 26)
for label, key in (("P@10", "precision"), ("R@10", "recall_at_cut"),
                   ("F1@10", "f1"), ("MAP", "map"),
                   ("nDCG@10", "ndcg"), ("R@100", "recall")):
    print(f"{label:<16}{train_record['macro'][key]:>10.4f}")
print("-" * 26)
print(f"{'fit':<16}{train_record['fit']['ndcg']:>10.4f}")
print(f"{'check':<16}{train_record['check']['ndcg']:>10.4f}")
print(f"{'topics':<16}{train_record['n_topics']:>10}")
print()
print("These are the numbers dev is about to be compared against. A large")
print("drop on dev means the choices were fitted to train.")

---

## 2. Chạy `dev`

Từ đây trở đi không quay lại chỉnh gì nữa.

Nếu hệ thống đã chọn có chặng GPU, chặng đó cần một gói việc **mới** cho `dev`:
câu truy vấn khác thì danh sách ứng viên khác. Ô lệnh bên dưới xuất gói đó khi
cần và dừng lại chờ.

In [ ]:
# --- Does dev need a GPU pass --------------------------------------------
needs_gpu = [stage for stage in P.STAGES
             if (CHOSEN.get(stage) or {}).get("kind") == "gpu_job"]

if needs_gpu:
    print(f"This system has GPU stages: {', '.join(needs_gpu)}")
    print()
    print("Each one names a job id whose scores were computed for the TRAIN")
    print("queries. dev has different queries, so it needs its own job.")
    print()
    print("Steps:")
    print("  1. run the non-GPU part on dev, to get its candidate list")
    print("  2. export a dev job, run the worker, import the scores")
    print("  3. point a copy of this system at the new job id")
    print()
    print("The cell below does step 1 and 2.")
else:
    print("This system runs entirely on the CPU. Nothing to hand off.")

In [ ]:
# --- Prepare the dev GPU job, if one is needed ---------------------------
GPU_JOBS = Path("..") / "gpu_jobs"
DEV_SUFFIX = "_dev"
dev_ready = True

if needs_gpu:
    GPU_JOBS.mkdir(exist_ok=True)
    # Run everything up to (not including) the first GPU stage.
    upstream = {k: v for k, v in CHOSEN.items()
                if k not in needs_gpu or k in ("name", "note")}
    upstream["name"] = CHOSEN["name"] + "_upstream_dev"
    for stage in needs_gpu:
        upstream[stage] = None

    if any(upstream.get(s) for s in P.STAGES):
        upstream_digest = P.run(upstream, split="dev")
        for stage in needs_gpu:
            job_id = CHOSEN[stage]["job"] + DEV_SUFFIX
            folder = GPU_JOBS / job_id
            if not (folder / "scores.jsonl").exists():
                P.export_gpu_job(upstream_digest, folder, job_id,
                                 task="rerank" if stage == "rerank" else "dense",
                                 split="dev", with_text=False)
                print(f"\nRun on the GPU machine:")
                print(f"    python gpu_worker.py {job_id} "
                      f"--data reteco_data/track1_tempo")
                dev_ready = False
            else:
                P.import_gpu_scores(folder / "scores.jsonl", job_id)
                print(f"{job_id}: scores already here")
    else:
        print("This system's first stage is itself a GPU job; export it "
              "directly for dev.")
        dev_ready = False

if dev_ready:
    print("\nReady to run dev.")
else:
    print("\nRun the worker, copy scores.jsonl back, then run this cell again.")

In [ ]:
# --- The one dev run ------------------------------------------------------
if dev_ready:
    DEV_SYSTEM = json.loads(json.dumps(CHOSEN))       # a copy, not the original
    DEV_SYSTEM["name"] = CHOSEN["name"] + "_dev"
    for stage in needs_gpu:
        DEV_SYSTEM[stage] = dict(CHOSEN[stage],
                                 job=CHOSEN[stage]["job"] + DEV_SUFFIX)

    started = time.time()
    dev_digest = P.run(DEV_SYSTEM, split="dev")
    dev_record = P.score(dev_digest, split="dev")
    print(f"\ntook {time.time() - started:.0f}s")
    print(f"hash: {dev_digest}")
    print()
    print(f"queries answered: {dev_record['n_topics']:,} of "
          f"{dev_record['n_gold_queries']:,}")
    if dev_record["n_topics"] != dev_record["n_gold_queries"]:
        print("WARNING: some dev queries came back empty. The scorer drops")
        print("those rather than scoring them zero, so this number is not")
        print("comparable to a system that answered all of them.")

---

## 3. Đủ bộ độ đo trên `dev`

Bảng này là kết quả chính của dự án. Cột `train` để cạnh cột `dev` để thấy
mức tụt.

In [ ]:
# --- train and dev, side by side -----------------------------------------
if dev_ready:
    METRICS = [("P@10", "precision"), ("R@10", "recall_at_cut"),
               ("F1@10", "f1"), ("MAP", "map"),
               ("nDCG@10", "ndcg"), ("R@100", "recall")]

    print(f"{'metric':<12}{'train':>10}{'dev':>10}{'change':>11}")
    print("-" * 43)
    for label, key in METRICS:
        a = train_record["macro"][key]
        b = dev_record["macro"][key]
        print(f"{label:<12}{a:>10.4f}{b:>10.4f}{b - a:>+11.4f}")
    print("-" * 43)
    print(f"{'topics':<12}{train_record['n_topics']:>10}"
          f"{dev_record['n_topics']:>10}")
    print()
    drop = train_record["macro"]["ndcg"] - dev_record["macro"]["ndcg"]
    if abs(drop) < 0.02:
        print(f"dev is within {abs(drop):.4f} of train. The choices made on")
        print("train carried over, which is what the fit/check discipline was")
        print("for.")
    else:
        direction = "below" if drop > 0 else "above"
        print(f"dev is {abs(drop):.4f} {direction} train. Worth reading the")
        print("per-domain table before drawing a conclusion: a single domain")
        print("moving can carry the macro average.")

In [ ]:
# --- Per domain on dev ----------------------------------------------------
if dev_ready:
    print(f"{'domain':<14}{'queries':>9}"
          + "".join(f"{label:>10}" for label, _ in METRICS))
    print("-" * (23 + 10 * len(METRICS)))
    for domain, cell in sorted(dev_record["per_domain"].items(),
                               key=lambda kv: -kv[1]["ndcg"]):
        row = f"{domain:<14}{cell['n_gold_queries']:>9}"
        row += "".join(f"{cell[key]:>10.4f}" for _, key in METRICS)
        print(row)
    print("-" * (23 + 10 * len(METRICS)))
    row = f"{'macro':<14}{dev_record['n_gold_queries']:>9}"
    row += "".join(f"{dev_record['macro'][key]:>10.4f}" for _, key in METRICS)
    print(row)

In [ ]:
# --- train against dev, per domain ---------------------------------------
if dev_ready:
    shared = [d for d in dev_record["per_domain"]
              if d in train_record["per_domain"]]
    pairs = sorted(((d, train_record["per_domain"][d]["ndcg"],
                     dev_record["per_domain"][d]["ndcg"]) for d in shared),
                   key=lambda r: -(r[2] - r[1]))

    fig, ax = plt.subplots(figsize=(9.4, 5.0))
    values = [dev - train for _, train, dev in pairs]
    bars = ax.barh([p[0] for p in pairs], values,
                   color=[TEAL if v >= 0 else ORANGE for v in values],
                   height=0.62)
    ax.invert_yaxis()
    ax.axvline(0, color=AXIS, linewidth=0.9)
    span = max(abs(v) for v in values) or 1
    ax.set_xlim(min(values) - span * 0.45, max(values) + span * 0.45)
    for bar, (domain, train_value, dev_value) in zip(bars, pairs):
        right = (dev_value - train_value) >= 0
        ax.text((dev_value - train_value) + span * (0.03 if right else -0.03),
                bar.get_y() + bar.get_height() / 2,
                f"{dev_value - train_value:+.4f}   "
                f"{train_value:.3f} -> {dev_value:.3f}",
                va="center", ha="left" if right else "right",
                fontsize=8.5, color=INK_SOFT)
    up = sum(1 for v in values if v > 0)
    finish(ax, f"dev cao hon train o {up}/{len(values)} nhom",
           subtitle=f"{CHOSEN['name']}, thay doi nDCG@10 giua hai tap",
           xlabel="dev tru train", grid_axis="x")
    plt.show()

---

## 4. Xuất file nộp

Định dạng TREC, sáu cột cách nhau bằng khoảng trắng:

```text
<ma cau truy van> Q0 <ma tai lieu> <thu hang> <diem> <ten he thong>
```

`P.submit` ghi file rồi gọi `format_checker.py` của ban tổ chức. Nếu bộ kiểm
định báo lỗi, hàm **xoá file đi** thay vì để lại một file hỏng trông như file
tốt.

In [ ]:
# --- Write and validate ---------------------------------------------------
if dev_ready:
    TAG = CHOSEN["name"]
    out_path = SUBMISSIONS / f"{TAG}_dev.txt"

    submitted = P.submit(DEV_SYSTEM, out_path, tag=TAG, split="dev",
                         checker=CHECKER if CHECKER.exists() else None,
                         top_k=10)
    print()
    if out_path.exists():
        lines = out_path.read_text(encoding="utf-8").splitlines()
        print(f"{out_path}  —  {len(lines):,} lines, "
              f"{out_path.stat().st_size / 1e6:.1f} MB")
        print()
        print("First three lines:")
        for line in lines[:3]:
            print(f"    {line}")
        print()
        by_query = {}
        for line in lines:
            by_query[line.split()[0]] = by_query.get(line.split()[0], 0) + 1
        counts = set(by_query.values())
        print(f"queries in the file: {len(by_query):,}")
        print(f"results per query:   {sorted(counts)}")
        if counts != {10}:
            print("  Not every query has 10 results. That is allowed, but it")
            print("  means some queries ran out of candidates.")
    else:
        print("The checker rejected the file, so it was deleted. Fix the")
        print("problem it reported and run this cell again.")

---

## 5. Danh sách phải công khai

Luật cuộc thi yêu cầu bài báo hệ thống liệt kê mọi mô hình và API đã dùng. Ô
lệnh dưới sinh danh sách đó từ chính cấu hình của hệ thống, nên nó không thể
quên mất thứ gì đã thật sự chạy.

In [ ]:
# --- The disclosure list, generated from the system itself ---------------
def describe(system):
    lines = []
    for stage in P.STAGES:
        config = system.get(stage)
        if not config:
            continue
        kind = config.get("kind")
        if kind == "sparse":
            lines.append(
                f"{stage}: {config['model']} (cai dat lai trong src/reteco.py), "
                f"tokenizer={config['tokenizer']}, "
                f"query_form={config['query_form']}, "
                f"k1={config.get('k1')}, b={config.get('b')}, "
                f"depth={config.get('depth')}")
        elif kind == "content_hash":
            lines.append(
                f"{stage}: khu trung lap theo bam noi dung, "
                f"expand={config.get('expand', 1)} (khong dung mo hinh nao)")
        elif kind == "gpu_job":
            job_file = GPU_JOBS / config["job"] / "job.json"
            model = "unknown"
            if job_file.exists():
                model = json.loads(job_file.read_text(encoding="utf-8")).get(
                    "model", "unknown")
            lines.append(f"{stage}: {model}  (job {config['job']})")
        elif kind == "rrf":
            lines.append(
                f"{stage}: reciprocal rank fusion, k={config.get('k')}, "
                f"gop voi {config.get('sources')} (khong dung mo hinh nao)")
        else:
            lines.append(f"{stage}: {kind}")
    return lines


print("MO HINH VA CONG CU DA DUNG")
print("=" * 60)
for line in describe(CHOSEN):
    print(f"  {line}")
print()
print("  Thu vien: numpy, scipy, matplotlib")
if any((CHOSEN.get(s) or {}).get("kind") == "gpu_job" for s in P.STAGES):
    print("            torch, sentence-transformers (chi o may GPU)")
print()
print("  Du lieu: chi dung kho van ban do ban to chuc cung cap.")
print("           Khong dung nguon ben ngoai nao.")
print()
print("  Ghi chu: notebook 01a phat hien ma tai lieu co cau truc de lo dau la")
print("           dap an. Phat hien nay duoc ghi lai trong bao cao va KHONG")
print("           duoc dua vao he thong du thi.")
print("=" * 60)

disclosure = {
    "system": CHOSEN["name"],
    "stages": describe(CHOSEN),
    "libraries": ["numpy", "scipy", "matplotlib"],
    "external_data": [],
    "id_leak_used": False,
}
(CACHE / "disclosure.json").write_text(json.dumps(disclosure, indent=1),
                                       encoding="utf-8")
print(f"\nSaved {CACHE / 'disclosure.json'}")

---

## 6. Bảng tổng của cả dự án

Từ hệ thống cơ sở tới hệ thống đem nộp.

In [ ]:
# --- The whole project, in one table -------------------------------------
MILESTONES = [
    ("He thong co so", 0.0719, "02"),
    ("BM25 chinh thuc", 0.0879, "starter kit"),
    ("Sau khi chon term", 0.1463, "03a"),
    ("Query Likelihood", 0.1839, "04 muc 11"),
    ("Sau khi tinh chinh k1, b", 0.1867, "05"),
]
if dev_ready:
    MILESTONES.append((f"He thong nop ({CHOSEN['name']}), train",
                       train_record["macro"]["ndcg"], "07d"))
    MILESTONES.append((f"He thong nop, dev",
                       dev_record["macro"]["ndcg"], "08"))

print(f"{'moc':<38}{'nDCG@10':>10}{'x co so':>10}{'o dau':>14}")
print("-" * 72)
baseline = MILESTONES[0][1]
for label, value, where in MILESTONES:
    print(f"{label:<38}{value:>10.4f}{value / baseline:>10.2f}{where:>14}")
print("-" * 72)

if dev_ready:
    fig, ax = plt.subplots(figsize=(9.6, 4.6))
    labels = [m[0] for m in MILESTONES]
    values = [m[1] for m in MILESTONES]
    ys = np.arange(len(labels))
    bars = ax.barh(ys, values, height=0.62,
                   color=[VIOLET if i >= len(values) - 2 else BLUE
                          for i in range(len(values))])
    ax.set_yticks(ys)
    ax.set_yticklabels(labels, fontsize=9.5)
    ax.invert_yaxis()
    ax.set_xlim(0, max(values) * 1.18)
    label_bars(ax, bars, values, fmt="{:.4f}")
    # Reference line: the official BM25 on dev, the only published number in
    # the same frame as the final bar. No target line -- no leaderboard exists.
    OFFICIAL_DEV = 0.0967
    ax.axvline(OFFICIAL_DEV, color=INK_MUTED, linewidth=0.9,
               linestyle=(0, (4, 3)))
    ax.annotate(f"BM25 chinh thuc tren dev  {OFFICIAL_DEV:.4f}",
                xy=(OFFICIAL_DEV, len(labels) - 0.4),
                xytext=(5, 0), textcoords="offset points",
                fontsize=8.5, color=INK_MUTED, va="center")
    finish(ax, f"Tu {baseline:.4f} len {dev_record['macro']['ndcg']:.4f} tren dev",
           subtitle="nDCG@10 trung binh theo 13 nhom",
           xlabel="nDCG@10", grid_axis="x")
    plt.show()

In [ ]:
# --- Everything, saved ----------------------------------------------------
if dev_ready:
    final = {
        "chosen_system": CHOSEN,
        "train": {"hash": train_digest, "macro": train_record["macro"],
                  "fit": train_record["fit"], "check": train_record["check"],
                  "n_topics": train_record["n_topics"]},
        "dev": {"hash": dev_digest, "macro": dev_record["macro"],
                "n_topics": dev_record["n_topics"],
                "per_domain": dev_record["per_domain"]},
        "submission": str(out_path),
        "milestones": [{"label": m[0], "ndcg": m[1], "where": m[2]}
                       for m in MILESTONES],
        "disclosure": disclosure,
    }
    (CACHE / "final_summary.json").write_text(json.dumps(final, indent=1),
                                              encoding="utf-8")
    print(f"Saved {CACHE / 'final_summary.json'}")
    print()
    print("Refresh the comparison page:")
    print("    python ../src/report.py --results results --open")
    print()
    P.table()

---

## 7. Kết luận

### `dev` đã dùng

Con số ở mục 3 là ước lượng không thiên lệch duy nhất mà dự án có. Muốn thử
thêm một ý tưởng nữa thì thử trên `train`, và con số `dev` ở trên không còn
dùng lại được cho hệ thống mới đó.

### Những gì đi kèm bài nộp

| Thứ | File |
| --- | --- |
| File nộp TREC | `submissions/` |
| Cấu hình hệ thống | `systems/<tên>.json` |
| Danh sách mô hình phải công khai | `results/disclosure.json` |
| Toàn bộ số liệu | `results/final_summary.json` |
| Trang so sánh | `results/report.html` |

Cấu hình hệ thống là một file JSON ngắn, và cùng với mã nguồn trong `src/` thì
nó đủ để người khác dựng lại đúng con số này. Đó là điều một bài báo hệ thống
cần chứng minh được.

### Điều dự án đã ghi lại mà không dùng

Mã tài liệu trong bộ dữ liệu có cấu trúc để lộ đâu là đáp án. Lọc theo cấu trúc
đó loại được 75–89% kho mà không mất đáp án nào. Dự án ghi lại phát hiện này
trong báo cáo và **không đưa vào hệ thống dự thi**, vì nó khai thác cách bộ dữ
liệu được tạo ra chứ không giải bài toán, và không có gì bảo đảm nó còn đúng
trên tập test ẩn.